# Project 1. Seq2Seq Num2Text
숫자·영어·한국어 6방향 번역 · GRU Encoder–Decoder + Attention · Greedy 디코딩

## 1. 환경 설정 및 데이터 파일

In [1]:
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
BATCH_SIZE = 256
EMB_DIM = 128
HIDDEN_DIM = 256
SINGLE_MAX_LEN = 100
BATCH_MAX_LEN = 48
MAX_EPOCHS = 15
EARLY_STOP_PATIENCE = 4
LEARNING_RATE = 1e-3
MODEL_DIR = Path.cwd()

LANGS = ('num', 'en', 'ko')
PAIRS = (('num', 'en'), ('num', 'ko'), ('en', 'ko'))
SPECIAL = ['<PAD>', '<SOS>', '<EOS>', '<UNK>', '<TO_num>', '<TO_en>', '<TO_ko>']
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    torch.set_num_threads(1)
print('PyTorch:', torch.__version__, '| Device:', device)

def find_files(names, folders):
    for folder in folders:
        if all((folder / name).exists() for name in names):
            return folder
    try:
        from google.colab import files
        print('Upload:', ', '.join(names))
        files.upload()
        if all(Path(name).exists() for name in names):
            return Path.cwd()
    except ImportError:
        pass
    raise FileNotFoundError('Place these files beside the notebook: ' + ', '.join(names))

DATA_DIR = find_files(
    ['num2txt.txt', 'num2kor.txt'],
    [Path.cwd(), Path.cwd().parent],
)

PyTorch: 2.14.1+cu130 | Device: cuda
Upload: num2txt.txt, num2kor.txt


Saving num2kor.txt to num2kor.txt
Saving num2txt.txt to num2txt.txt


## 2. 데이터 병합·분할 (`0` 행 Train 이동)

In [2]:
english = pd.read_csv(
    DATA_DIR / 'num2txt.txt', sep='\t', header=None,
    names=['num', 'en'], dtype=str, keep_default_na=False,
)
korean = pd.read_csv(
    DATA_DIR / 'num2kor.txt', sep='\t', header=None,
    names=['num', 'ko'], dtype=str, keep_default_na=False,
)
assert english.num.is_unique and korean.num.is_unique
assert english.num.str.fullmatch(r'\d+').all() and korean.num.str.fullmatch(r'\d+').all()
df = english.merge(korean, on='num', how='inner', validate='one_to_one')
assert len(df) == 100000

original_train, temporary = train_test_split(df, test_size=0.20, random_state=SEED)
original_val, test_df = train_test_split(temporary, test_size=0.5, random_state=SEED)
zero_row = original_val[original_val.num == '0'].copy()
assert zero_row.to_dict('records') == [{'num': '0', 'en': 'zero', 'ko': '영'}]
train_df = pd.concat([original_train, zero_row], ignore_index=True)
val_df = original_val[original_val.num != '0'].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
for first, second in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(first.num).isdisjoint(set(second.num))
assert len(train_df) == 80001 and len(val_df) == 9999 and len(test_df) == 10000
print('Train / Validation / Test:', len(train_df), len(val_df), len(test_df))

Train / Validation / Test: 80001 9999 10000


## 3. 토큰화 및 Vocabulary

In [3]:
def tokenize(text, language):
    if language not in LANGS:
        raise ValueError(language)
    return str(text).split() if language == 'en' else list(str(text))

def detokenize(tokens, language):
    return ' '.join(tokens) if language == 'en' else ''.join(tokens)

def prepare_input(text, source_lang, target_lang, token_to_id):
    tokens = [f'<TO_{target_lang}>'] + tokenize(text, source_lang) + ['<EOS>']
    return torch.tensor([token_to_id.get(token, UNK_ID) for token in tokens], dtype=torch.long)

def prepare_target(text, target_lang, token_to_id):
    tokens = tokenize(text, target_lang) + ['<EOS>']
    return torch.tensor([token_to_id.get(token, UNK_ID) for token in tokens], dtype=torch.long)

counter = Counter()
for language in LANGS:
    for text in train_df[language]:
        counter.update(tokenize(text, language))
itos = SPECIAL + sorted(token for token in counter if token not in SPECIAL)
stoi = {token: index for index, token in enumerate(itos)}
assert len(itos) == 65 and 'zero' in stoi and '영' in stoi
print('Vocabulary:', len(itos), '| zero tokens:', {'zero': stoi['zero'], '영': stoi['영']})

Vocabulary: 65 | zero tokens: {'zero': 49, '영': 57}


## 4. 모델: GRU Encoder → Attention → Decoder

In [4]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src, lengths):
        embedded = self.embedding(src)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False,
        )
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=src.size(1),
        )
        return outputs, hidden, src.ne(PAD_ID)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
        self.scale = hidden_dim ** -0.5

    def forward(self, tokens, hidden, enc_outputs, enc_mask):
        embedded = self.embedding(tokens)
        dec_outputs, hidden = self.gru(embedded, hidden)
        scores = torch.bmm(dec_outputs, enc_outputs.transpose(1, 2)) * self.scale
        scores = scores.masked_fill(~enc_mask.unsqueeze(1), -1e9)
        attention = torch.softmax(scores, dim=-1)
        context = torch.bmm(attention, enc_outputs)
        logits = self.fc(torch.cat([dec_outputs, context], dim=-1))
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, emb_dim, hidden_dim)

    def forward(self, src, lengths, tgt):
        enc_outputs, hidden, mask = self.encoder(src, lengths)
        start = torch.full((tgt.size(0), 1), SOS_ID, dtype=torch.long, device=tgt.device)
        decoder_inputs = torch.cat([start, tgt[:, :-1]], dim=1)
        logits, _ = self.decoder(decoder_inputs, hidden, enc_outputs, mask)
        return logits

    @torch.no_grad()
    def greedy_decode(self, src, lengths, max_len=100):
        self.eval()
        enc_outputs, hidden, mask = self.encoder(src, lengths)
        token = torch.full((src.size(0), 1), SOS_ID, dtype=torch.long, device=src.device)
        generated = [[] for _ in range(src.size(0))]
        finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)

        for _ in range(max_len):
            logits, hidden = self.decoder(token, hidden, enc_outputs, mask)
            next_token = logits[:, -1, :].argmax(dim=-1)
            for row, token_id in enumerate(next_token.tolist()):
                if not finished[row].item() and token_id != EOS_ID:
                    generated[row].append(token_id)
            finished |= next_token.eq(EOS_ID)
            if finished.all().item():
                break
            token = next_token.unsqueeze(1)
        return generated

## 5. 학습

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, frame, first_lang, second_lang, vocabulary):
        token_to_id = {token: index for index, token in enumerate(vocabulary)}
        self.examples = []
        for first, second in frame[[first_lang, second_lang]].itertuples(index=False, name=None):
            directions = (
                (first, first_lang, second, second_lang),
                (second, second_lang, first, first_lang),
            )
            for source, source_lang, target, target_lang in directions:
                source_ids = prepare_input(source, source_lang, target_lang, token_to_id)
                target_ids = prepare_target(target, target_lang, token_to_id)
                assert UNK_ID not in source_ids.tolist() + target_ids.tolist()
                self.examples.append((source_ids, target_ids))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

def collate_fn(batch):
    source_ids, target_ids = zip(*batch)
    return (
        pad_sequence(source_ids, batch_first=True, padding_value=PAD_ID),
        torch.tensor([len(source) for source in source_ids], dtype=torch.long),
        pad_sequence(target_ids, batch_first=True, padding_value=PAD_ID),
    )

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    for source, lengths, target in loader:
        source, target = source.to(device), target.to(device)
        with torch.set_grad_enabled(training):
            logits = model(source, lengths, target)
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), target.reshape(-1), ignore_index=PAD_ID,
            )
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        token_count = target.ne(PAD_ID).sum().item()
        total_loss += loss.item() * token_count
        total_tokens += token_count
    return total_loss / max(total_tokens, 1)

def train_pair_from_scratch(first_lang, second_lang, save_dir, epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE, lr=LEARNING_RATE):
    save_dir = Path(save_dir)
    save_dir.mkdir(exist_ok=True, parents=True)
    train_loader = DataLoader(
        TranslationDataset(train_df, first_lang, second_lang, itos),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        TranslationDataset(val_df, first_lang, second_lang, itos),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
    )
    model = Seq2Seq(len(itos), EMB_DIM, HIDDEN_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss = float('inf')
    stale_epochs = 0
    history = []
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader)
        history.append({'pair': f'{first_lang}_{second_lang}', 'epoch': epoch,
                        'train_loss': train_loss, 'validation_loss': val_loss})
        print(first_lang, second_lang, epoch, round(train_loss, 6), round(val_loss, 6),
              '| seconds:', round(time.perf_counter() - started, 2))
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            stale_epochs = 0
            torch.save({
                'model_state_dict': model.state_dict(), 'itos': itos,
                'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM,
                'pair': (first_lang, second_lang), 'epoch': epoch, 'best_val_loss': best_loss,
            }, save_dir / f'{first_lang}_{second_lang}.pt')
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break
    return history

checkpoint_paths = [MODEL_DIR / f'{first}_{second}.pt' for first, second in PAIRS]
existing_paths = [path for path in checkpoint_paths if path.is_file()]

if len(existing_paths) == len(checkpoint_paths):
    print('세 개의 체크포인트가 모두 있습니다. 재학습을 건너뛰고 저장 모델을 재로드합니다.')
    for path in checkpoint_paths:
        print('  Reuse:', path.name)
    history_path = MODEL_DIR / 'training_history.csv'
    if history_path.is_file():
        history_df = pd.read_csv(history_path)
        print('기존 학습 기록:', history_path.name)
    else:
        history_df = pd.DataFrame()
elif existing_paths:
    missing_paths = [path.name for path in checkpoint_paths if not path.is_file()]
    raise FileNotFoundError(
        '체크포인트가 일부만 있습니다. 기존 모델을 보호하기 위해 자동 학습하지 않습니다. '
        '누락된 파일을 추가하거나 새 빈 폴더에서 실행하세요. 누락: '
        + ', '.join(missing_paths)
    )
else:
    all_histories = []
    for first_lang, second_lang in PAIRS:
        print('\nTraining', first_lang, '↔', second_lang, flush=True)
        history = train_pair_from_scratch(first_lang, second_lang, MODEL_DIR)
        all_histories.extend(history)

    history_df = pd.DataFrame(all_histories)
    history_df.to_csv(MODEL_DIR / 'training_history.csv', index=False)
    display(history_df.groupby('pair').tail(1))
    print('학습한 체크포인트와 학습 기록 저장:', MODEL_DIR)


Training num ↔ en
num en 1 0.472155 0.02669 | seconds: 10.79
num en 2 0.01464 0.007054 | seconds: 7.53
num en 3 0.005344 0.003378 | seconds: 8.2
num en 4 0.002803 0.002196 | seconds: 8.53
num en 5 0.005806 0.00161 | seconds: 7.75
num en 6 0.000983 0.001214 | seconds: 8.38
num en 7 0.000891 0.001799 | seconds: 8.44
num en 8 0.001164 0.001452 | seconds: 7.58
num en 9 0.001097 0.001145 | seconds: 8.17
num en 10 0.000838 0.000997 | seconds: 7.92
num en 11 0.000718 0.001541 | seconds: 7.9
num en 12 0.001318 0.000689 | seconds: 8.28
num en 13 0.00034 0.000783 | seconds: 7.69
num en 14 0.000591 0.000434 | seconds: 8.27
num en 15 0.000533 0.001055 | seconds: 8.15

Training num ↔ ko
num ko 1 0.417773 0.018723 | seconds: 9.75
num ko 2 0.008645 0.007936 | seconds: 9.31
num ko 3 0.00426 0.001493 | seconds: 9.43
num ko 4 0.002023 0.000885 | seconds: 9.66
num ko 5 0.003994 0.001222 | seconds: 9.65
num ko 6 0.000897 0.000503 | seconds: 9.4
num ko 7 0.000381 0.000368 | seconds: 9.37
num ko 8 0.000707

,pair,epoch,train_loss,validation_loss
14,num_en,15,0.000533,0.001055
29,num_ko,15,0.000198,0.000152
44,en_ko,15,0.000281,0.000434


학습한 체크포인트와 학습 기록 저장: /content


## 6. 저장 모델 로드

In [6]:
def build_model(checkpoint, device='cpu'):
    model = Seq2Seq(
        len(checkpoint['itos']), checkpoint['emb_dim'], checkpoint['hidden_dim'],
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    return model.eval()

def load_models(model_dir):
    loaded = {}
    for first_lang, second_lang in PAIRS:
        checkpoint = torch.load(
            Path(model_dir) / f'{first_lang}_{second_lang}.pt',
            map_location='cpu', weights_only=True,
        )
        assert checkpoint['itos'] == itos, 'Checkpoint vocabulary/order differs.'
        assert tuple(checkpoint['pair']) == (first_lang, second_lang)
        loaded[frozenset((first_lang, second_lang))] = build_model(checkpoint, device)
    return loaded

models = load_models(MODEL_DIR)
print('Loaded model pairs:', [f'{first}↔{second}' for first, second in PAIRS])

Loaded model pairs: ['num↔en', 'num↔ko', 'en↔ko']


## 7. Greedy 번역 함수

In [7]:
@torch.no_grad()
def predict_batch(model, texts, source_lang, target_lang, vocabulary,
                  *, batch_size=BATCH_SIZE, max_len=BATCH_MAX_LEN):
    token_to_id = {token: index for index, token in enumerate(vocabulary)}
    all_ids = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        sequences = [prepare_input(text, source_lang, target_lang, token_to_id)
                     for text in batch_texts]
        lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
        padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
        padded = padded.to(next(model.parameters()).device)
        decoded = model.greedy_decode(padded, lengths, max_len=max_len)
        all_ids.extend(decoded)
    raw_texts = [
        detokenize([vocabulary[index] for index in ids if vocabulary[index] not in SPECIAL], target_lang)
        for ids in all_ids
    ]
    return {'raw_texts': raw_texts}

def translate(model, text, source_lang, target_lang, itos, *, trim_outer_whitespace=True):
    result = predict_batch(
        model, [str(text)], source_lang, target_lang, itos,
        batch_size=1, max_len=SINGLE_MAX_LEN,
    )
    raw_text = result['raw_texts'][0]
    return raw_text.strip() if trim_outer_whitespace else raw_text

def translate_final(text, source_lang, target_lang, *, trim_outer_whitespace=True):
    if source_lang not in LANGS or target_lang not in LANGS or source_lang == target_lang:
        raise ValueError('Choose two different languages from num, en, ko.')
    model = models[frozenset((source_lang, target_lang))]
    return translate(model, text, source_lang, target_lang, itos,
                     trim_outer_whitespace=trim_outer_whitespace)

## 8. Validation / Test 평가

In [8]:
@torch.no_grad()
def eval_split(frame, split_name):
    rows = frame.to_dict('records')
    summaries = []
    errors = []
    for first_lang, second_lang in PAIRS:
        model = models[frozenset((first_lang, second_lang))]
        for source_lang, target_lang in [(first_lang, second_lang), (second_lang, first_lang)]:
            predicted = predict_batch(
                model, [row[source_lang] for row in rows], source_lang, target_lang, itos,
            )['raw_texts']
            raw_errors = 0
            normalized_errors = 0
            for row, raw_text in zip(rows, predicted):
                raw_errors += raw_text != row[target_lang]
                normalized_errors += raw_text.strip() != row[target_lang]
                if raw_text.strip() != row[target_lang]:
                    errors.append({
                        'direction': f'{source_lang}→{target_lang}', 'num': row['num'],
                        'source': row[source_lang], 'predicted': raw_text.strip(),
                        'expected': row[target_lang],
                    })
            summaries.append({
                'split': split_name, 'direction': f'{source_lang}→{target_lang}', 'total': len(rows),
                'raw_errors': raw_errors, 'normalized_errors': normalized_errors,
                'normalized_em': (len(rows) - normalized_errors) / len(rows),
            })
            print(f'{source_lang}→{target_lang}: {normalized_errors} errors / {len(rows)}', flush=True)
    return pd.DataFrame(summaries), pd.DataFrame(errors)

val_scores, val_errors = eval_split(val_df, 'Validation')
test_scores, remaining_errors = eval_split(test_df, 'Test')
display(val_scores)
display(test_scores)
print('Test error examples (max 20):')
display(remaining_errors.head(20))
print('All Test error rows:', len(remaining_errors))
print('Test raw / trimmed errors:',
      test_scores.raw_errors.sum(), test_scores.normalized_errors.sum())

num→en: 10 errors / 9999
en→num: 4 errors / 9999
num→ko: 4 errors / 9999
ko→num: 1 errors / 9999
en→ko: 2 errors / 9999
ko→en: 4 errors / 9999
num→en: 5 errors / 10000
en→num: 6 errors / 10000
num→ko: 3 errors / 10000
ko→num: 0 errors / 10000
en→ko: 3 errors / 10000
ko→en: 2 errors / 10000


,split,direction,total,raw_errors,normalized_errors,normalized_em
0,Validation,num→en,9999,10,10,0.9990
1,Validation,en→num,9999,4,4,0.9996
2,Validation,num→ko,9999,4,4,0.9996
3,Validation,ko→num,9999,1,1,0.9999
4,Validation,en→ko,9999,2,2,0.9998
5,Validation,ko→en,9999,4,4,0.9996


,split,direction,total,raw_errors,normalized_errors,normalized_em
0,Test,num→en,10000,5,5,0.9995
1,Test,en→num,10000,6,6,0.9994
2,Test,num→ko,10000,5,3,0.9997
3,Test,ko→num,10000,0,0,1.0000
4,Test,en→ko,10000,3,3,0.9997
5,Test,ko→en,10000,2,2,0.9998


Test error examples (max 20):


,direction,num,source,predicted,expected
0,num→en,5060081002,5060081002,five billion sixty million eight thousand two,five billion sixty million eighty one thousand...
1,num→en,5860005667,5860005667,five billion eight hundred sixty million five ...,five billion eight hundred sixty million five ...
2,num→en,400800212,400800212,four hundred million eight thousand two hundre...,four hundred million eight hundred thousand tw...
3,num→en,8000169,8000169,eight million ten thousand sixteen,eight million one hundred sixty nine
4,num→en,5586700007,5586700007,five billion five hundred eighty six million s...,five billion five hundred eighty six million s...
5,en→num,615000457,six hundred fifteen million four hundred fifty...,61504570,615000457
6,en→num,7609000774,seven billion six hundred nine million seven h...,760900074,7609000774
7,en→num,2713000396,two billion seven hundred thirteen million thr...,2713030960,2713000396
8,en→num,8000344,eight million three hundred forty four,800034,8000344
9,en→num,803001004,eight hundred three million one thousand four,803000100,803001004


All Test error rows: 19
Test raw / trimmed errors: 21 19


## 9. `0` 번역 확인

In [9]:
zero_example = {'num': '0', 'en': 'zero', 'ko': '영'}
zero_checks = []
for first_lang, second_lang in PAIRS:
    for source_lang, target_lang in [(first_lang, second_lang), (second_lang, first_lang)]:
        predicted = translate_final(zero_example[source_lang], source_lang, target_lang)
        zero_checks.append({
            'direction': f'{source_lang}→{target_lang}',
            'source': zero_example[source_lang],
            'expected': zero_example[target_lang],
            'predicted': predicted,
            'correct': predicted == zero_example[target_lang],
        })
zero_check_df = pd.DataFrame(zero_checks)
display(zero_check_df)
print('Zero training-example checks:', int(zero_check_df.correct.sum()), '/ 6')

,direction,source,expected,predicted,correct
0,num→en,0,zero,zero,True
1,en→num,zero,0,0,True
2,num→ko,0,영,영,True
3,ko→num,영,0,0,True
4,en→ko,zero,영,영,True
5,ko→en,영,zero,zero,True


Zero training-example checks: 6 / 6


## 10. 저장 모델 재로드 및 테스트

In [10]:
# 사용 방법
# 1) 이 노트북, num_en.pt / num_ko.pt / en_ko.pt, 데이터 num2txt.txt / num2kor.txt를 같은 폴더에 둔다.
# 2) 위 셀을 순서대로 실행한 뒤 이 셀을 실행하면 저장된 .pt를 새로 불러와 6개 방향을 테스트한다.
# 3) 임의 입력 번역: translate_final('12708', 'num', 'en'), translate_final('만 이천칠백팔', 'ko', 'num')
#    언어 코드는 'num', 'en', 'ko' / 숫자도 문자열로 입력

models = load_models(MODEL_DIR)
for example in [
    {'num': '0', 'en': 'zero', 'ko': '영'},
    test_df.iloc[0][['num', 'en', 'ko']].to_dict(),
]:
    print('\nSource number:', example['num'])
    for first_lang, second_lang in PAIRS:
        for source_lang, target_lang in [(first_lang, second_lang), (second_lang, first_lang)]:
            predicted = translate_final(example[source_lang], source_lang, target_lang)
            print(source_lang, '→', target_lang, '|', repr(predicted),
                  '| match:', predicted == example[target_lang])
            assert isinstance(predicted, str), "Expected translated text."
print('\nPASS: saved models reloaded and all six translation directions executed.')


Source number: 0
num → en | 'zero' | match: True
en → num | '0' | match: True
num → ko | '영' | match: True
ko → num | '0' | match: True
en → ko | '영' | match: True
ko → en | 'zero' | match: True

Source number: 657456544
num → en | 'six hundred fifty seven million four hundred fifty six thousand five hundred forty four' | match: True
en → num | '657456544' | match: True
num → ko | '육억 오천칠백사십오만 육천오백사십사' | match: True
ko → num | '657456544' | match: True
en → ko | '육억 오천칠백사십오만 육천오백사십사' | match: True
ko → en | 'six hundred fifty seven million four hundred fifty six thousand five hundred forty four' | match: True

PASS: saved models reloaded and all six translation directions executed.
